<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Import Modules
    <\h1>
</span>

In [1]:
%config Completer.use_jedi = False

import platform
from datetime import datetime
import numpy as np
import pandas as pd
from funcs import *

pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.2f}".format

<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Set Project Path
    <\h1>
</span>

In [2]:
PROJECT_PATH = r"C:\thesis\Shirazi_Code\Landscape_Evapotranspiration"


<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Read Raw Data
    <\h1>
</span>

In [3]:
raw_data = pd.read_excel(PROJECT_PATH + r"\assets\input\Raw_Data_20210216.xlsx", 
                         engine = "openpyxl")

location_name = pd.read_excel(PROJECT_PATH + r"\assets\input\Location_Name_20210417.xlsx",
                              engine = "openpyxl")

raw_coeff = pd.read_excel(PROJECT_PATH + r"\assets\input\Landscape_Coefficient_20210318.xlsx",
                          sheet_name=None,
                          engine = "openpyxl")

date = datetime.today().strftime('%Y%m%d')

<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Data Cleansing
    <\h1>
</span>

In [4]:
data, data_duplicates, data_meters_errors, data_duplicates_id = data_cleansing(data=raw_data,
                                                                               location_name=location_name)

file_name = PROJECT_PATH + r"\assets\output\data_errors_" + date + ".xlsx"
with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="w") as writer:
    data_duplicates.to_excel(writer, index=False, sheet_name="data_duplicates")
    data_meters_errors.to_excel(writer, index=False, sheet_name="data_meters_errors")
    data_duplicates_id.to_excel(writer, index=False, sheet_name="data_duplicates_id")
        
file_name = PROJECT_PATH + r"\assets\output\data_cleansing_" + date + ".xlsx"
with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="w") as writer:
    data.to_excel(writer, index=False, sheet_name="Sheet1")

<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Tree Area Modify
    <\h1>
</span>

In [5]:
data = tree_area_modify(data=data)

file_name = PROJECT_PATH + r"/assets/report/dataset_" + date + ".xlsx"
with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="w") as writer:
    data.to_excel(writer, index=False, sheet_name="Sheet1")

c:\thesis\Shirazi_Code\Landscape_Evapotranspiration\notebooks\funcs.py:301: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, df])


<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        Area Report
    <\h1>
</span>

In [6]:
report = report_area(data=data)

file_name = PROJECT_PATH + r"\assets\report\area_report_" + date + ".xlsx"
with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="w") as writer:
    report.to_excel(writer, index=True, sheet_name="لکه")

report_noSpot = report.reset_index().drop(["نام نوع لکه", "نام لکه"], axis=1)

with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="a") as writer:
    report_noSpot.groupby(['منطقه', 'ناحیه', 'پیمان']).sum().to_excel(writer, index=True, sheet_name="پیمان")

with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="a") as writer:
    report_noSpot.groupby(['منطقه', 'ناحیه']).sum().to_excel(writer, index=True, sheet_name="ناحیه")

with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="a") as writer:
    report_noSpot.groupby(['منطقه']).sum().to_excel(writer, index=True, sheet_name="منطقه")

c:\thesis\Shirazi_Code\Landscape_Evapotranspiration\notebooks\funcs.py:310: FutureWarning: The provided callable <function sum at 0x000001F5BBCB8B80> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  tmp1 = data.pivot_table(
c:\thesis\Shirazi_Code\Landscape_Evapotranspiration\notebooks\funcs.py:320: FutureWarning: The provided callable <function mean at 0x000001F5BBCB9F80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  tmp2 = data.pivot_table(


<span style='color:blue; text-align:left; font-family:Britannic'>
    <h1>
        NoGreen Area
    <\h1>
</span>

In [7]:
df = report.reset_index()[["منطقه", "نام نوع لکه", "مساحت آدرس ریزآیتمی", "مجموع سطوح غیر سبز"]]
df.columns = ["منطقه", "نوع لکه", "مساحت لکه", "مساحت معابر"]
df['مساحت لکه'] = df['مساحت لکه'].replace(0, np.nan)
df = df.dropna(subset=['مساحت لکه'])
df['مساحت لکه'] = df['مساحت لکه'] / 10000
df['مساحت معابر'] = df['مساحت معابر'] / 10000
# df["نسبت معابر"] = (df["مساحت معابر"] / df["مساحت لکه"]) * 100
result = pd.pivot_table(df,
#                         values=['نسبت معابر', 'مساحت لکه', 'مساحت معابر'],
                        values=['مساحت لکه', 'مساحت معابر'],
                        index=['منطقه'],
                        columns=['نوع لکه'],
                        aggfunc={
                            "مساحت لکه" : np.sum,
                            "مساحت معابر" : np.sum,
#                             "نسبت معابر" : np.mean
                                },
                        fill_value=0).reset_index()


file_name = PROJECT_PATH + r"/assets/report/noGreen_" + date + ".xlsx"
with pd.ExcelWriter(path=file_name, engine="openpyxl", mode="w") as writer:
    result.to_excel(writer, index=True, sheet_name="مساحت غیرسبز")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_4572\1469238506.py:8: FutureWarning: The provided callable <function sum at 0x000001F5BBCB8B80> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  result = pd.pivot_table(df,
C:\Users\ASUS\AppData\Local\Temp\ipykernel_4572\1469238506.py:8: FutureWarning: The provided callable <function sum at 0x000001F5BBCB8B80> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  result = pd.pivot_table(df,


In [8]:
data.query('`نام زیرمجموعه هر قلم`=="متر مربع"').query('`نوع آیتم`=="ریزآیتمی"').groupby(["منطقه", "نام نوع لکه"])["مقدار آیتم"].sum().reset_index()\
.pivot(index='منطقه', columns='نام نوع لکه', values=["مقدار آیتم"]).reset_index()

منطقه مقدار آیتم                                          \
نام نوع لکه         آیلند ها آیلند های بزرگراه جنگل کاری داخل محدوده   
0              01  82,313.00               NaN                   NaN   
1              02 181,763.00         68,430.00            169,327.00   
2              03  75,392.00         58,331.00             13,714.00   
3              04  29,678.00         22,698.00                   NaN   
4              05  53,137.00         64,672.00                   NaN   
5              06 103,074.00        144,957.00                   NaN   
6              07 162,044.00        173,248.00          2,208,572.00   
7              08  38,035.00         21,891.00             57,717.00   
8              09 245,779.00        168,308.00          2,095,349.00   
9              10 102,774.00         96,835.00             13,684.00   
10             11 109,909.00         30,205.00            100,552.00   
11             12 142,541.00         60,338.00            199,675.00   
12             13  11,967.00               NaN                   NaN   

                                                                 \
نام نوع لکه حاشیه معابر حاشیه های بزرگراه    لچکی ها     میادین   
0            137,557.00               NaN  15,167.00   6,580.00   
1             98,190.00        106,209.00  70,675.00  76,869.00   
2             47,159.00         37,598.00   5,493.00  38,997.00   
3             43,614.00         25,705.00   2,531.00  35,271.00   
4             39,876.00          7,154.00   9,318.00   3,858.00   
5             43,413.00          8,000.00  43,668.00  10,888.00   
6            231,360.00        150,152.00  29,396.00  62,429.00   
7            164,911.00         92,900.00   6,858.00  18,855.00   
8            166,238.00        135,304.00   2,141.00  62,368.00   
9            143,918.00        261,311.00 127,483.00 218,952.00   
10           107,724.00         35,440.00  36,156.00  29,213.00   
11           187,768.00         47,211.00  40,892.00  21,414.00   
12            12,995.00               NaN   3,305.00   4,923.00   

                                                                \
نام نوع لکه پارک های بالای 10 هکتار پارک های بین 6 تا 10 هکتار   
0                        108,195.00                        NaN   
1                        153,392.00                   6,810.00   
2                        429,266.00                        NaN   
3                        275,760.00                        NaN   
4                         78,472.00                        NaN   
5                        132,802.00                  42,424.00   
6                      1,073,499.00                 290,738.00   
7                        289,687.00                        NaN   
8                        507,663.00                  81,346.00   
9                               NaN                        NaN   
10                       458,755.00                        NaN   
11                              NaN                        NaN   
12                              NaN                        NaN   

                                                    
نام نوع لکه پارک های زیر 6 هکتار کمربند سبز حفاظتی  
0                      76,067.00               NaN  
1                     314,221.00               NaN  
2                     329,915.00               NaN  
3                      83,793.00               NaN  
4                     190,311.00               NaN  
5                     260,398.00               NaN  
6                     374,151.00         27,270.00  
7                      71,756.00        350,118.00  
8                     399,015.00               NaN  
9                     575,711.00               NaN  
10                    169,171.00               NaN  
11                    200,555.00               NaN  
12                     39,047.00               NaN

In [9]:
df = data.query('`نوع آیتم`=="ریزآیتمی"').query('`زیرمجموعه هر قلم` == ["01", "02", "03", "04"]').groupby(["منطقه", "نام نوع قلم", "نام زیرمجموعه هر قلم"])["مقدار آیتم خام"].sum().reset_index()\
# .pivot(index='منطقه', columns=["نام زیرمجموعه هر قلم", "نام نوع قلم"], values=["مقدار آیتم خام"]).reset_index()

df.query('`نام نوع قلم` != "درخت و درختچه"').groupby(["منطقه", "نام زیرمجموعه هر قلم"])["مقدار آیتم خام"].sum().reset_index()\
.pivot(index='منطقه', columns=["نام زیرمجموعه هر قلم"], values=["مقدار آیتم خام"]).reset_index()


منطقه مقدار آیتم خام                              \
نام زیرمجموعه هر قلم        آبیاری تانکری آبیاری تحت فشار آبیاری ثقلی   
0                       01      24,135.00       44,965.00         NaN   
1                       02      79,732.00      137,868.00         NaN   
2                       03      17,583.00      197,791.00      506.00   
3                       04       7,951.00      155,070.00         NaN   
4                       05       8,131.00      129,780.00         NaN   
5                       06      13,893.00       59,551.00         NaN   
6                       07      68,349.00      199,069.00    3,045.00   
7                       08       8,597.00      123,685.00   10,000.00   
8                       09      39,962.00      226,562.00    1,770.00   
9                       10       3,331.00      219,930.00         NaN   
10                      11      45,106.00      264,369.00    3,170.00   
11                      12       4,806.00       74,840.00         NaN   
12                      13       5,334.00        4,498.00         NaN   

                                   
نام زیرمجموعه هر قلم آبیاری شلنگی  
0                      310,770.00  
1                      575,367.00  
2                      495,122.00  
3                      237,076.00  
4                      228,545.00  
5                      523,651.00  
6                      561,447.00  
7                      273,552.00  
8                      779,114.00  
9                      649,061.00  
10                     582,102.00  
11                     280,793.00  
12                      60,900.00